# FreshGuard v2 - 05 Evaluate

Inputs required on Kaggle:
- `freshguard-v2-splits`
- `freshguard-v2-detector-data`
- `freshguard-v2-detector-artifacts`
- `freshguard-v2-classifier-artifacts`
- `freshguard-official-sources-v2`
- `ulnnproject/food-freshness-dataset`
- optional five-apple image dataset

Food Freshness test metrics are the canonical 24-class freshness benchmark. KTH GroceryStoreDataset official test split is reported as type-only external evidence.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from fnmatch import fnmatch
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

def log(message: str) -> None:
    print(f'[{time.strftime("%H:%M:%S")}] {message}', flush=True)

log('Importing Ultralytics YOLO')
try:
    from ultralytics import YOLO
except ImportError:
    log('Ultralytics is not installed; installing ultralytics>=8.4.0,<9.0.0')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.4.0,<9.0.0'], check=True)
    from ultralytics import YOLO
log('Ultralytics import is ready')

KAGGLE_INPUT = Path('/kaggle/input')

def input_roots() -> list[Path]:
    roots: list[Path] = []
    for child in sorted(p for p in KAGGLE_INPUT.iterdir() if p.is_dir()):
        roots.append(child)
        if child.name in {'datasets', 'notebooks'}:
            for owner in sorted(p for p in child.iterdir() if p.is_dir()):
                for item in sorted(p for p in owner.iterdir() if p.is_dir()):
                    roots.append(item)
    seen: set[str] = set()
    unique_roots: list[Path] = []
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            unique_roots.append(root)
    return unique_roots

def walk_for_marker(root: Path, pattern: str, max_depth: int = 4) -> Path | None:
    skip_dirs = {'.git', '__pycache__', 'images', 'labels', 'Dataset', 'dataset', 'Fresh', 'Rotten', 'train', 'val', 'test', 'sample_images'}
    for dirpath, dirnames, filenames in os.walk(root):
        current = Path(dirpath)
        depth = len(current.relative_to(root).parts)
        if depth >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [name for name in dirnames if name not in skip_dirs]
        for filename in filenames:
            if fnmatch(filename, pattern):
                return current / filename
    return None

def first_match(pattern: str, preferred_terms: tuple[str, ...] = ()) -> Path | None:
    roots = input_roots()
    preferred = [root for root in roots if any(term in str(root).lower() for term in preferred_terms)]
    fallback = [root for root in roots if root not in preferred]
    log(f'Searching {len(preferred)} preferred roots for {pattern}')
    for root in preferred:
        match = walk_for_marker(root, pattern)
        if match is not None:
            log(f'Found {pattern}: {match}')
            return match
    log(f'Searching {len(fallback)} fallback roots for {pattern}')
    for root in fallback:
        match = walk_for_marker(root, pattern)
        if match is not None:
            log(f'Found {pattern}: {match}')
            return match
    log(f'Found {pattern}: None')
    return None

def require_marker(label: str, pattern: str, preferred_terms: tuple[str, ...]) -> Path:
    match = first_match(pattern, preferred_terms)
    if match is None:
        attached_roots = [str(root) for root in input_roots()]
        nearby_files: list[str] = []
        for root in input_roots():
            if any(term in str(root).lower() for term in preferred_terms):
                for dirpath, dirnames, filenames in os.walk(root):
                    current = Path(dirpath)
                    if len(current.relative_to(root).parts) >= 3:
                        dirnames[:] = []
                    nearby_files.extend(str(current / filename) for filename in filenames[:20])
                    if len(nearby_files) >= 60:
                        break
        raise FileNotFoundError(f'Missing required input {label!r}; could not find {pattern!r} under /kaggle/input. Attached roots: {attached_roots}. Nearby files in matching roots: {nearby_files[:60]}')
    return match

log('Starting required-input preflight checks')
preflight = {
    'freshguard-v2-splits': require_marker('freshguard-v2-splits', 'classifier_manifest_food_freshness_only.csv', ('split', 'freshguard-v2-splits', 'freshguard_v2_splits')),
    'freshguard-v2-detector-data': require_marker('freshguard-v2-detector-data', 'produce.yaml', ('detector-data', 'detector_data')),
    'freshguard-v2-detector-artifacts': require_marker('freshguard-v2-detector-artifacts', 'yolo26n_produce_v2.pt', ('detector-artifact', 'detector_artifact', 'package-detector', 'package_detector')),
    'freshguard-v2-classifier-artifacts': require_marker('freshguard-v2-classifier-artifacts', 'dinov3_vits16_food_freshness_v2.pt', ('classifier', 'dinov3')),
    'freshguard-official-sources-v2': require_marker('freshguard-official-sources-v2', 'official_sources_summary.json', ('official', 'sources')),
}
log('Searching for Food Freshness Dataset/Fresh and Dataset/Rotten tree')
food_dataset_roots = sorted(
    p for p in KAGGLE_INPUT.rglob('Dataset')
    if p.is_dir() and (p / 'Fresh').exists() and (p / 'Rotten').exists()
)
if not food_dataset_roots:
    raise FileNotFoundError("Missing Food Freshness Dataset input; could not find a Dataset/Fresh + Dataset/Rotten tree under /kaggle/input")
preflight['ulnnproject/food-freshness-dataset'] = food_dataset_roots[0]
log(f'Found Food Freshness root: {food_dataset_roots[0]}')

food_manifests = [preflight['freshguard-v2-splits']]
SPLITS_DIR = food_manifests[0].parent if food_manifests else None
produce_yamls = [preflight['freshguard-v2-detector-data']]
DETECTOR_DATA_DIR = produce_yamls[0].parent if produce_yamls else None
DETECTOR_ARTIFACT = preflight['freshguard-v2-detector-artifacts']
CLASSIFIER_ARTIFACT = preflight['freshguard-v2-classifier-artifacts']
APPLE_SMOKE_DIR = next((p for p in KAGGLE_INPUT.glob('*apple*') if p.is_dir()), None)
OUT_DIR = Path('/kaggle/working/freshguard_v2_eval')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for name, value in {'splits': SPLITS_DIR, 'detector_data': DETECTOR_DATA_DIR, 'detector_artifact': DETECTOR_ARTIFACT, 'classifier_artifact': CLASSIFIER_ARTIFACT}.items():
    if value is None:
        raise FileNotFoundError(f'Missing required v2 evaluation input: {name}')

log('Checking 250 Food Freshness manifest image paths')
food_probe = pd.read_csv(SPLITS_DIR / 'classifier_manifest_food_freshness_only.csv', usecols=['path']).head(250)
missing_food = [path for path in food_probe['path'].tolist() if not Path(path).exists()]
if missing_food:
    raise FileNotFoundError(f'Food Freshness input is attached but manifest paths are not readable. First missing paths: {missing_food[:5]}')
grocery_probe_path = SPLITS_DIR / 'grocery_external_eval_manifest.csv'
if grocery_probe_path.exists():
    log('Checking 250 KTH GroceryStoreDataset manifest image paths')
    grocery_probe = pd.read_csv(grocery_probe_path, usecols=['path']).head(250)
    missing_grocery = [path for path in grocery_probe['path'].tolist() if not Path(path).exists()]
    if missing_grocery:
        raise FileNotFoundError(f'Official sources input is attached but KTH manifest paths are not readable. First missing paths: {missing_grocery[:5]}')
log('Preflight path checks passed')
DETECTOR_DATA_YAML = DETECTOR_DATA_DIR / 'produce.yaml'
RUNTIME_DETECTOR_DATA_YAML = Path('/kaggle/working/produce_eval_runtime.yaml')
yaml_lines = DETECTOR_DATA_YAML.read_text().splitlines()
yaml_lines = [f'path: {DETECTOR_DATA_DIR}' if line.strip().startswith('path:') else line for line in yaml_lines]
RUNTIME_DETECTOR_DATA_YAML.write_text('\n'.join(yaml_lines) + '\n')
print({'preflight': {name: str(path) for name, path in preflight.items()}, 'splits': str(SPLITS_DIR), 'detector_data': str(DETECTOR_DATA_DIR), 'detector_runtime_yaml': str(RUNTIME_DETECTOR_DATA_YAML), 'detector': str(DETECTOR_ARTIFACT), 'classifier': str(CLASSIFIER_ARTIFACT), 'device': str(DEVICE)})

In [ ]:
log('Loading classifier checkpoint')
checkpoint = torch.load(CLASSIFIER_ARTIFACT, map_location=DEVICE, weights_only=True)
CLASS_NAMES = tuple(checkpoint['class_names'])
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
MODEL_NAME = checkpoint.get('model_name', 'vit_small_patch16_dinov3')
IMAGE_SIZE = int(checkpoint.get('image_size', 256))
HEAD_HIDDEN_DIM = int(checkpoint.get('head_hidden_dim', 512))

class DinoV3FreshnessModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.backbone = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
        self.head = nn.Sequential(nn.Linear(int(self.backbone.num_features), HEAD_HIDDEN_DIM), nn.GELU(), nn.Dropout(p=0.1), nn.Linear(HEAD_HIDDEN_DIM, len(CLASS_NAMES)))

    def forward(self, tensor: torch.Tensor) -> torch.Tensor:
        return self.head(self.backbone(tensor))

log('Building classifier model')
model = DinoV3FreshnessModel().to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
log('Classifier model is ready')
eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC), transforms.ToTensor(), transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])

In [ ]:
class FreshnessDataset(Dataset):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.frame.iloc[index]
        image = Image.open(row.path).convert('RGB')
        return eval_tf(image), torch.tensor(CLASS_TO_INDEX[row.combined_label], dtype=torch.long)

def predict_loader(loader: DataLoader, label: str) -> tuple[list[int], list[int]]:
    y_true: list[int] = []
    y_pred: list[int] = []
    total_batches = len(loader)
    log(f'Starting {label}: {total_batches} batches')
    with torch.no_grad():
        for batch_index, (images, labels) in enumerate(loader, start=1):
            logits = model(images.to(DEVICE, non_blocking=True))
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())
            y_true.extend(labels.tolist())
            if batch_index == 1 or batch_index % 25 == 0 or batch_index == total_batches:
                log(f'{label}: completed {batch_index}/{total_batches} batches')
    return y_true, y_pred

log('Loading Food Freshness canonical test manifest')
food_df = pd.read_csv(SPLITS_DIR / 'classifier_manifest_food_freshness_only.csv')
food_test = food_df[(food_df['split'] == 'test') & (food_df['combined_label'].isin(CLASS_TO_INDEX))].copy()
food_loader = DataLoader(FreshnessDataset(food_test), batch_size=96, shuffle=False, num_workers=2, pin_memory=True)
food_true, food_pred = predict_loader(food_loader, 'Food Freshness classifier eval')
food_report = classification_report(food_true, food_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
classifier_metrics = {'macro_f1': float(f1_score(food_true, food_pred, average='macro')), 'top1_accuracy': float(accuracy_score(food_true, food_pred)), 'per_class': {name: food_report[name] for name in CLASS_NAMES}}
print({k: v for k, v in classifier_metrics.items() if k != 'per_class'})

In [ ]:
grocery_external_metrics: dict[str, object] = {'available': False}
grocery_path = SPLITS_DIR / 'grocery_external_eval_manifest.csv'
if grocery_path.exists():
    log('Starting KTH GroceryStoreDataset external type-only eval')
    grocery = pd.read_csv(grocery_path)
    correct = 0
    total = 0
    rows: list[dict[str, object]] = []
    with torch.no_grad():
        for row_index, row in enumerate(grocery.itertuples(index=False), start=1):
            image = Image.open(row.path).convert('RGB')
            tensor = eval_tf(image).unsqueeze(0).to(DEVICE)
            probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
            label = CLASS_NAMES[int(np.argmax(probs))]
            pred_type = label.rsplit('_', maxsplit=1)[0]
            expected_type = str(row.produce_type)
            correct += int(pred_type == expected_type)
            total += 1
            rows.append({'path': row.path, 'expected_type': expected_type, 'predicted_label': label, 'predicted_type': pred_type, 'confidence': float(probs.max())})
            if row_index == 1 or row_index % 100 == 0 or row_index == len(grocery):
                log(f'KTH type-only eval: completed {row_index}/{len(grocery)} images')
    grocery_external_metrics = {'available': True, 'type_accuracy': float(correct / max(1, total)), 'count': int(total)}
    pd.DataFrame(rows).to_csv(OUT_DIR / 'grocery_external_type_predictions.csv', index=False)
print(grocery_external_metrics)

In [ ]:
log('Loading YOLO detector artifact')
detector = YOLO(str(DETECTOR_ARTIFACT))
log('Starting YOLO detector validation on test split')
detector_val = detector.val(data=str(RUNTIME_DETECTOR_DATA_YAML), split='test', imgsz=1024)
detector_metrics = {'conf_0.40': {'overall': {'mAP50': float(detector_val.box.map50), 'mAP50_95': float(detector_val.box.map)}}}
print(detector_metrics)

In [ ]:
def predict_image(path: Path) -> dict[str, object]:
    image = Image.open(path).convert('RGB')
    tensor = eval_tf(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    order = np.argsort(probs)[::-1]
    return {'path': str(path), 'prediction': CLASS_NAMES[int(order[0])], 'confidence': float(probs[int(order[0])]), 'top2_margin': float(probs[int(order[0])] - probs[int(order[1])])}

apple_smoke = []
if APPLE_SMOKE_DIR is not None:
    smoke_paths = sorted(p for p in APPLE_SMOKE_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'})
    log(f'Starting five-apple smoke eval: {len(smoke_paths)} images')
    for index, path in enumerate(smoke_paths, start=1):
        apple_smoke.append(predict_image(path))
        log(f'Apple smoke eval: completed {index}/{len(smoke_paths)} images')
print({'apple_smoke_count': len(apple_smoke)})

In [ ]:
log('Writing eval_report.json and eval_report.md')
eval_report = {
    'classifier_only': classifier_metrics,
    'grocery_external_type_only': grocery_external_metrics,
    'detector': detector_metrics,
    'apple_smoke': apple_smoke,
    'contract': {'classes': list(CLASS_NAMES), 'detector_checkpoint': Path(DETECTOR_ARTIFACT).name, 'classifier_checkpoint': Path(CLASSIFIER_ARTIFACT).name, 'classifier_model_name': MODEL_NAME, 'classifier_image_size': IMAGE_SIZE},
}
(OUT_DIR / 'eval_report.json').write_text(json.dumps(eval_report, indent=2))
md = ['# FreshGuard v2 Evaluation Report', '', '## Food Freshness Canonical 24-Class Test', '', f"- Macro F1: **{classifier_metrics['macro_f1']:.4f}**", f"- Top-1 accuracy: {classifier_metrics['top1_accuracy']:.4f}", '', '## KTH GroceryStoreDataset External Type-Only Test', '', f"- Available: {grocery_external_metrics['available']}", f"- Type accuracy: {grocery_external_metrics.get('type_accuracy', 'n/a')}", f"- Count: {grocery_external_metrics.get('count', 0)}", '', '## Detector', '', f"- mAP50: {detector_metrics['conf_0.40']['overall']['mAP50']:.4f}", f"- mAP50-95: {detector_metrics['conf_0.40']['overall']['mAP50_95']:.4f}", '', '## Five-Apple Smoke Set', '']
if apple_smoke:
    md.extend(f"- `{Path(item['path']).name}` -> `{item['prediction']}` ({item['confidence']:.3f})" for item in apple_smoke)
else:
    md.append('- Smoke images were not attached to this Kaggle run.')
(OUT_DIR / 'eval_report.md').write_text('\n'.join(md) + '\n')
print({'json': str(OUT_DIR / 'eval_report.json'), 'markdown': str(OUT_DIR / 'eval_report.md')})